# Collection Discovery: searching for collections across multiple APIs using the Federated STAC Collection Discovery API

Author: Henry Rodman (Development Seed)

Date: October 27, 2025

Description: These examples show how to use the Federated Collection Discovery STAC API to search for collections across multiple STAC APIs. There is also an interactive search application for using the API which you can use at [https://discover.maap-project.org](https://discover.maap-project.org).

## Background
It can be challenging to find the data that you need for an analysis when any of the following are true:
- you don't know the collection ID for a collection that you know exists
- you don't know which exact API the data can be accessed from
- you don't know which collections you even need

Fear not! The Federated STAC Collection Discovery application (and the underlying API) can help you find the data you need by running your search for collections across multiple catalogs simultaneously.

![Federated Collection Discovery application](./federated-collection-discovery-app.png)

## Using [discover.maap-project.org](https://discover.maap-project.org)

The Federated Collection Discovery web application is a great place to browse for datasets that may be relevant to your work. The application allows you to apply free-text, spatial, and temporal filters to the collections that are housed in a set of configured catalogs. By default, the application will search through the following catalogs:

* NASA MAAP STAC: https://stac.maap-project.org/
* ESA MAAP STAC: https://catalog.maap.eo.esa.int/catalogue/
* VEDA STAC: https://openveda.cloud/api/stac/
* NASA CMR STAC: https://cmr.earthdata.nasa.gov/stac/ALL

You can toggle any of these APIs on/off and add new STAC APIs to the list in the 'Settings' menu:

<img src="./settings.png" width="40%">

<div class="alert alert-block alert-warning">
<b>Note:</b> Any custom STAC API that you provide must have the free-text and collection-search STAC API extensions enabled. If those extensions are not available then the application will warn you that those search features are not available.
</div>

The search form allows you to apply text, temporal, and spatial filters to the collections:

<img src="./search-form.png" width="40%">

### text search
Perform a search with a free-text filter for collections that include 'elevation' OR 'DEM' but not 'biomass'. The API will scan the 'title', 'description', and 'keywords' attributes of all of the collections in the catalogs.

The free-text query parameter will follow the logic outlined in the [STAC API free-text extension](https://github.com/stac-api-extensions/freetext-search?tab=readme-ov-file). Here is a table that outlines the types of queries that are possible (borrowed from the STAC API free-text extension readme):
| q | Summary | Detail |
| ----------- | ------- | ------ |
| `sentinel` | Free-text query against all properties | This will search for any matching items that CONTAIN `"sentinel"` |
| `"climate model"` | Free-text search using exact | This will search for any matching items that CONTAIN the exact phrase `"climate model"` |
|`climate model`| Using `OR` term match (**Default**) | This will search for any matching items that CONTAIN `"climate"` OR `"model"`|
|`climate OR model`| Using `OR` term match (**Default**) | This will search for any matching items that CONTAIN `"climate"` OR `"model"`|
|`climate AND model`| Using `AND` term match | This will search for any matching items that CONTAIN `"climate"` AND `"model"`|
| `(quick OR brown) AND fox` | Parentheses can be used to group terms | This will search for matching items that CONTAIN `"quick"` OR `"brown"` AND `"fox"` |
| `quick +brown -fox` | Indicate included and excluded terms using `+`/`-` | This will search for items that INCLUDES `"brown"` EXCLUDES `"fox"` OR CONTAIN `"quick"` |

### spatial search
You can apply a spatial filter to your search by typing the bounding box coordinates (xmin, ymin, xmax, ymax) or by drawing one on the provided map interface. The bounding box filter will return collections where the spatial extent intersects the provided bounding box.

<img src="./spatial-search.png" width="30%">

### temporal search
You can apply a temporal filter to your search by entering a start/end date range in the provided input boxes.

### Inspect the search results
The matching collections from your search query will be printed in a table. If you click on a row, the collection details will pop up showing the description, the spatial and temporal extents, the source catalog, and data provider entries.

<img src="./collection-details.png" width="50%">

If the collection seems like a good match for your needs, you can scroll down to the 'STAC Item Code Hints' to get the Python and R code you need to start an item-level search in a notebook or script.

<img src="./code-hint.png" width="50%">


## Federated STAC Collection Discovery API
The Federated STAC Collection Discovery API is a STAC API that can be used to run collection searches across multiple upstream STAC APIs simultaneously.

For fully capable upstream STAC APIs, the following parameters are available on the `/collections` endpoint:

**search parameters**:
- `bbox`: bounding box coordinates (EPSG:4326)
- `datetime`: datetime extent
- `q`: free-text search
- `filter`: cql2 filtering
- `sortby`: sort results by some field

<div class="alert alert-block alert-info">
<b>Note:</b> The API uses the collection-search STAC API extension which can be paired with several other extensions to create a robust search interface for collections across multiple STAC APIs. However, the functionality of the Federated STAC Collection Discovery API is limited to the capabilities of the least capable upstream API. For example, if you try to use it with an upstream STAC API that does not implement the collection-search extension or the free-text search extension, the advanced search capabilities will not be available at all.
</div>

In [29]:
from datetime import datetime, timezone

import httpx
import pandas as pd
from pystac import RelType
from pystac_client import Client
from IPython.display import display, HTML
from requests import Request

API_URL = "https://discover-api.maap-project.org"

The API is configured to search across several STAC APIs by default, you can use the `/_mgmt/health` endpoint to see the collection-search capabilities to see the collection-search capabilities of each of the configured upstream STAC APIs.

In [2]:
api_status = httpx.get(f"{API_URL}/_mgmt/health", timeout=20).json()
api_status

{'status': 'UP',
 'lifespan': {'status': 'UP'},
 'upstream_apis': {'https://stac.maap-project.org': {'healthy': True,
   'collection_search_conformance': ['collection-search',
    'collection-search#fields',
    'collection-search#filter',
    'collection-search#free-text',
    'collection-search#query',
    'collection-search#sort']},
  'https://staging.openveda.cloud/api/stac': {'healthy': True,
   'collection_search_conformance': ['collection-search',
    'collection-search#fields',
    'collection-search#filter',
    'collection-search#free-text',
    'collection-search#query',
    'collection-search#sort']},
  'https://catalog.maap.eo.esa.int/catalogue': {'healthy': True,
   'collection_search_conformance': ['collection-search',
    'collection-search#filter',
    'collection-search#free-text']},
  'https://cmr.earthdata.nasa.gov/stac/ALL': {'healthy': True,
   'collection_search_conformance': ['collection-search',
    'collection-search#free-text',
    'collection-search#sort']}}

## Searching for collections with pystac-client

Since the Federated Collection Discovery API is a valid STAC API we can use pystac-client to run searches and interact with the results.

In [34]:
client = Client.open(API_URL)

search = client.collection_search(
    q="GEDI",
    limit=1,
    max_collections=4,
)

for i, collection in enumerate(search.collections()):
    print(f"\n- {collection.id}")
    print(f"  Title: {collection.title}")
    print(f"  Upstream API link: {collection.get_self_href()}")
    description = (
        collection.description[:100] + "..."
        if len(collection.description) > 100
        else collection.description
    )
    print(f"  Description: {description}")


- GEDI_CalVal_Field_Data
  Title: Global Ecosystem Dynamics Investigation (GEDI) Calibration/Validation Field Survey Dataset
  Upstream API link: https://stac.maap-project.org/collections/GEDI_CalVal_Field_Data
  Description: The Global Ecosystem Dynamics Investigation (GEDI) Forest Structure and Biomass Database (FSBD) is a...

- GEDI_ISS_L3_Canopy_Height_Mean_RH100_201904-202303
  Title: GEDI Mean Canopy Height AGL
  Upstream API link: https://staging.openveda.cloud/api/stac/collections/GEDI_ISS_L3_Canopy_Height_Mean_RH100_201904-202303
  Description: The Global Ecosystem Dynamics Investigation ([GEDI](https://gedi.umd.edu/)) mission aims to characte...

- C2142776747-LPCLOUD
  Title: GEDI L2B Canopy Cover and Vertical Profile Metrics Data Global Footprint Level V002
  Upstream API link: https://catalog.maap.eo.esa.int/catalogue/collections/C2142776747-LPCLOUD
  Description: The Global Ecosystem Dynamics Investigation ([GEDI](https://gedi.umd.edu/)) mission aims to characte...

- GE

### Searching Specific APIs with pystac-client

You can also specify which upstream APIs to query by supplying the `apis` parameter in a `request_modifier` function:

In [35]:
def add_apis(request: Request):
    """Add the `apis` parameter to all requests to this STAC API client"""
    request.params.update(
        {
            "apis": [
                "https://stac.eoapi.dev",
                "https://stac.maap-project.org",
            ]
        }
    )
    return request


specific_client = Client.open(API_URL, request_modifier=add_apis)

# Search for biomass collections from specific APIs only
search = specific_client.collection_search(
    q="biomass",
    limit=1,
    max_collections=4,
)

print("Searching specific APIs for biomass collections:")

# Show collection details
for i, collection in enumerate(search.collections()):
    print(f"\n- {collection.id}")
    print(f"  Title: {collection.title}")
    print(f"  Upstream API link: {collection.get_self_href()}")
    description = (
        collection.description[:100] + "..."
        if len(collection.description) > 100
        else collection.description
    )
    print(f"  Description: {description}")

Searching specific APIs for biomass collections:

- AFRISAR_DLR
  Title: AFRISAR_DLR
  Upstream API link: https://stac.maap-project.org/collections/AFRISAR_DLR
  Description: The  ESA  BIOMASS  mission  was  selected  in  2013  as  the  7th  Earth  Explorer  mission.  BIOMAS...

- AFRISAR_DLR2
  Title: AFRISAR_DLR2
  Upstream API link: https://stac.maap-project.org/collections/AFRISAR_DLR2
  Description: The ESA BIOMASS mission was selected in 2013 as the 7th Earth Explorer mission. BIOMASS will provide...

- AfriSAR_UAVSAR_Geocoded_Covariance
  Title: AfriSAR UAVSAR Geocoded Covariance Matrix product Generated Using NISAR Tools
  Upstream API link: https://stac.maap-project.org/collections/AfriSAR_UAVSAR_Geocoded_Covariance
  Description: The Geocoded Covariance Matrix dataset is the 4x4 Native Covariance Matrix geocoded to a spatial res...

- AfriSAR_UAVSAR_Geocoded_SLC
  Title: AfriSAR UAVSAR Geocoded SLCs Generated Using NISAR Tools
  Upstream API link: https://stac.maap-project.or

## Searching for collections using an http client



In [3]:
search_request = httpx.get(
    f"{API_URL}/collections",
    params={
        "q": "(elevation OR DEM) -biomass",
    },
    timeout=20,
)
search_request.raise_for_status()
search_results = search_request.json()

results_df = (
    pd.DataFrame(search_results["collections"])
    .assign(
        catalog_url=lambda df: df["links"].apply(
            lambda links_list: next(
                (link["href"] for link in links_list if link["rel"] == "root"), 
                None
            )
        )
    )
)
display(HTML(results_df[["id", "catalog_url", "title"]].to_html()))

,id,catalog_url,title
0,ABoVE_UAVSAR_PALSAR,https://stac.maap-project.org/,Arctic-Boreal Vulnerability Experiment Uninhabited Aerial Vehicle Synthetic Aperture Radar Polarimetric SAR
1,SRTMGL1_COD,https://stac.maap-project.org/,NASA Shuttle Radar Topography Mission Global 1
2,la-fires-slope,https://staging.openveda.cloud/api/stac/,Eaton and Palisades Fires (2025) Slope
3,la-fires-slope-test-march6,https://staging.openveda.cloud/api/stac/,Eaton and Palisades Fires (2025) Slope-Test-March6


The `results` contain a list of collection-level metadata with some basic properties that you can review further.

In [4]:
collection_info = results_df.iloc[0]
print(collection_info)

print("\ndescription:\n", collection_info.description)

id                                                      ABoVE_UAVSAR_PALSAR
type                                                             Collection
links                     [{'rel': 'items', 'type': 'application/geo+jso...
title                     Arctic-Boreal Vulnerability Experiment Uninhab...
extent                    {'spatial': {'bbox': [[-166.788382, 59.729364,...
license                                                             CC0-1.0
providers                 [{'url': 'https://above.nasa.gov/', 'name': 'N...
description               The Arctic-Boreal Vulnerability Experiment (AB...
item_assets                                                              {}
stac_version                                                          1.1.0
keywords                                                                NaN
stac_extensions                                                         NaN
assets                                                                  NaN
renders     

### bounding box filter
Perform a search for collections that intersect Finland's bounding box with a free-text filter for 'biomass'

In [5]:
finland_bbox = (18.061, 59.348, 31.181, 70.576)
search_request = httpx.get(
    f"{API_URL}/collections",
    params={
        "q": "biomass",
        "bbox": ",".join(str(coord) for coord in finland_bbox),
    },
    timeout=20,
)
search_request.raise_for_status()
search_results = search_request.json()

results_df = (
    pd.DataFrame(search_results["collections"])
    .assign(
        catalog_url=lambda df: df["links"].apply(
            lambda links_list: next(
                (link["href"] for link in links_list if link["rel"] == "root"), 
                None
            )
        )
    )
)
display(HTML(results_df[["id", "catalog_url", "title"]].to_html()))

,id,catalog_url,title
0,BIOSAR1,https://stac.maap-project.org/,BIOSAR1
1,ESACCI_Biomass_L4_AGB_V4_100m,https://stac.maap-project.org/,ESA CCI Above-Ground Biomass Product Level 4 Version 4
2,GEDI_CalVal_Field_Data,https://stac.maap-project.org/,Global Ecosystem Dynamics Investigation (GEDI) Calibration/Validation Field Survey Dataset
3,GEDI_CalVal_Lidar_Data_Compressed,https://stac.maap-project.org/,Global Ecosystem Dynamics Investigation (GEDI) Calibration/Validation Airborne Lidar Dataset (Compressed)
4,icesat2-boreal,https://stac.maap-project.org/,Circumpolar boreal forest structure from ICESat-2 & HLS (2020 v1.0): 30m aboveground woody biomass density
5,ICESat2_Boreal_AGB_tindex_average,https://stac.maap-project.org/,ICESat2-Boreal Above Ground Biomass T-Index Average
6,icesat2-boreal-v2.1-agb,https://stac.maap-project.org/,Circumpolar boreal forest structure from ICESat-2 & HLS (2020 v2.1): 30m aboveground woody biomass density
7,icesat2-boreal-v2.1-ht,https://stac.maap-project.org/,Circumpolar boreal forest structure from ICESat-2 & HLS (2020 v2.1): 30m vegetation height
8,icesat2-boreal-v3.0-agb,https://stac.maap-project.org/,Circumpolar boreal forest structure from ICESat-2 & HLS (2020 v3.0): 30m aboveground woody biomass density
9,icesat2-boreal-v3.0-ht,https://stac.maap-project.org/,Circumpolar boreal forest structure from ICESat-2 & HLS (2020 v3.0): 30m vegetation height


## pagination

The results returned by the API are paginated, so you can use the `next` link from the `/collections` response to access the next page. Pagination state is handled with the `token` parameter.

In [6]:
next_link = next((link["href"] for link in search_results["links"] if link["rel"] == "next"), None)
print(next_link)
next_results = httpx.get(next_link).json()

results_df = (
    pd.DataFrame(next_results["collections"])
    .assign(
        catalog_url=lambda df: df["links"].apply(
            lambda links_list: next(
                (link["href"] for link in links_list if link["rel"] == "root"), 
                None
            )
        )
    )
)
display(HTML(results_df[["id", "catalog_url", "title"]].to_html()))

https://discover-api.maap-project.org/collections?token=eyJjdXJyZW50Ijp7Imh0dHBzOi8vc3RhYy5tYWFwLXByb2plY3Qub3JnIjoiaHR0cHM6Ly9zdGFjLm1hYXAtcHJvamVjdC5vcmcvY29sbGVjdGlvbnM_YmJveD0xOC4wNjElMkM1OS4zNDglMkMzMS4xODElMkM3MC41NzYmbGltaXQ9MTAmZmlsdGVyX2xhbmc9Y3FsMi10ZXh0JnE9YmlvbWFzcyZvZmZzZXQ9MTAiLCJodHRwczovL2NhdGFsb2cubWFhcC5lby5lc2EuaW50L2NhdGFsb2d1ZSI6Imh0dHBzOi8vY2F0YWxvZy5tYWFwLmVvLmVzYS5pbnQvY2F0YWxvZ3VlL2NvbGxlY3Rpb25zP2Jib3g9MTguMDYxLDU5LjM0OCwzMS4xODEsNzAuNTc2JmxpbWl0PTEwJmZpbHRlcl9sYW5nPWNxbDItdGV4dCZxPWJpb21hc3Mmc3RhcnRSZWNvcmQ9MTEiLCJodHRwczovL2Ntci5lYXJ0aGRhdGEubmFzYS5nb3Yvc3RhYy9BTEwiOiJodHRwczovL2Ntci5lYXJ0aGRhdGEubmFzYS5nb3Yvc3RhYy9BTEwvY29sbGVjdGlvbnM_YmJveD0xOC4wNjElMkM1OS4zNDglMkMzMS4xODElMkM3MC41NzYmZmlsdGVyX2xhbmc9Y3FsMi10ZXh0JmxpbWl0PTEwJnE9YmlvbWFzcyZjdXJzb3I9ZXlKcWMyOXVJam9pV3pFdU5Dd3dMakFzTUM0d0xERTJNRGswTlRreE9UazVPVGtzWENJeklDMGdaM0pwWkdSbFpDQnZZbk5sY25aaGRHbHZibk5jSWl4Y0lrTk5VMTlIYkc5aVlXeGZSbTl5WlhOMFgwRm5aVjh5TXpRMVhDSXNYQ0l4WENJc016QTVNVEUxTXpNM09Td3hORjBpTENK

,id,catalog_url,title
0,icesat2-boreal-v3.1-agb,https://stac.maap-project.org/,Circumpolar boreal forest structure from ICESat-2 & HLS (2020 v3.1): 30m aboveground woody biomass density
1,icesat2-boreal-v3.1-ht,https://stac.maap-project.org/,Circumpolar boreal forest structure from ICESat-2 & HLS (2020 v3.1): 30m vegetation height
2,BiomassLevel2bIOC,https://catalog.maap.eo.esa.int/catalogue/,Biomass Level 2B (IOC)
3,BiomassLevel0,https://catalog.maap.eo.esa.int/catalogue/,Biomass Level 0
4,CCIBiomassV5.01,https://catalog.maap.eo.esa.int/catalogue/,CCI Biomass V5.01
5,BiomassLevel2b,https://catalog.maap.eo.esa.int/catalogue/,Biomass Level 2B
6,BiomassAuxIOC,https://catalog.maap.eo.esa.int/catalogue/,Biomass Auxiliary (IOC)
7,BiomassLevel0IOC,https://catalog.maap.eo.esa.int/catalogue/,Biomass Level 0 (IOC)
8,BiomassLevel1a,https://catalog.maap.eo.esa.int/catalogue/,Biomass Level 1A
9,C2756302505-ORNL_CLOUD,https://catalog.maap.eo.esa.int/catalogue/,"Aboveground Biomass Density for High Latitude Forests from ICESat-2, 2020"


## temporal filter
You can use the `datetime` parameter to filter down to collections with temporal extents that overlap a provided range. For example, to find collections with a temporal extent that includes the term 'spectral' and has data as recent as September 15, 2024, you can run the following search:

In [7]:
recent_date = datetime(year=2024, month=9, day=15, tzinfo=timezone.utc)

search_request = httpx.get(
    f"{API_URL}/collections",
    params={
        "datetime": f"{recent_date.isoformat().replace('+00:00', 'Z')}/..",
        "q": "spectral",
    },
    timeout=20,
)
search_request.raise_for_status()
search_results = search_request.json()

results_df = (
    pd.DataFrame(search_results["collections"])
    .assign(
        catalog_url=lambda df: df["links"].apply(
            lambda links_list: next(
                (link["href"] for link in links_list if link["rel"] == "root"), 
                None
            )
        )
    )
)
display(HTML(results_df[["id", "catalog_url", "title"]].to_html()))

,id,catalog_url,title
0,EOP:ESA:Sentinel-2,https://catalog.maap.eo.esa.int/catalogue/,Sentinel-2
1,ResourceSat-2.archive.and.tasking,https://catalog.maap.eo.esa.int/catalogue/,ResourceSat-2 full archive and tasking
2,EarthCAREL0L1Products,https://catalog.maap.eo.esa.int/catalogue/,EarthCARE L0 and L1 Products for the Commissioning Team
3,WorldView-3.full.archive.and.tasking,https://catalog.maap.eo.esa.int/catalogue/,WorldView-3 full archive and tasking
4,WorldView-2.full.archive.and.tasking,https://catalog.maap.eo.esa.int/catalogue/,WorldView-2 full archive and tasking
5,Vision-1.full.archive.and.tasking,https://catalog.maap.eo.esa.int/catalogue/,Vision-1 full archive and tasking
6,Pleiades.HiRI.archive.and.new,https://catalog.maap.eo.esa.int/catalogue/,Pleiades full archive and tasking
7,EarthCAREL1InstChecked_MAAP,https://catalog.maap.eo.esa.int/catalogue/,EarthCARE L1 Products for Cal/Val Users
8,EarthCAREL0L1Products_MAAP,https://catalog.maap.eo.esa.int/catalogue/,EarthCARE L0 and L1 Products for the Commissioning Team
9,EarthCAREL1Validated_MAAP,https://catalog.maap.eo.esa.int/catalogue/,EarthCARE L1 Products


<div class="alert alert-block alert-info">
<b>Note:</b> For open datetime ranges, use .. to represent either the beginning or ending timestamp.
</div>

## Specify APIs with `urls`
You can specify a specific set of STAC APIs to search through with the `apis` parameter. This will override the default STAC API URLs.

In [ ]:
stac_api_urls = [
    "https://stac.eoapi.dev",
]
search_request = httpx.get(
    f"{API_URL}/collections",
    params={
        "apis": ",".join(stac_api_urls),
        "q": "fire"
    },
    timeout=30,
)
search_request.raise_for_status()
search_results = search_request.json()

results_df = (
    pd.DataFrame(search_results["collections"])
    .assign(
        catalog_url=lambda df: df["links"].apply(
            lambda links_list: next(
                (link["href"] for link in links_list if link["rel"] == "root"), 
                None
            )
        )
    )
)
display(HTML(results_df[["id", "catalog_url", "title"]].to_html()))

<div class="alert alert-block alert-info">
<b>Note:</b> The collection discovery API will only provide search functionality that is provided by ALL of the upstream APIS. If one of the APIs that you want tos earch does not provide the free-text extension, then the `q` parameter will not do anything!
</div>

## Additional resources
- [Federated Collection Discovery source code](https://github.com/developmentseed/federated-collection-discovery)
- [Federated Collection Discovery app](https://discover.maap-project.org)
- [Federated Collection Discovery API docs](https://discover-api.maap-project.org/api.html)
- [STAC FastAPI Collection Discovery User Guide](https://developmentseed.org/stac-fastapi-collection-discovery/v0.2.3/using-the-api/)